CMI

In [5]:
import pandas as pd
import numpy as np

# Load file
df = pd.read_csv(
    "/content/drive/MyDrive/TreeGOER_CMI_2024.txt",
    sep='|',
    engine='python'
)

# Updated mapping of alphabet columns to climate types
climate_map = {
    "A": "very wet",
    "B": "wet",
    "C": "moist",
    "D": "dry sub-humid",
    "E": "semi-arid",
    "F": "arid",
    "G": "hyper-arid"
}

def classify_climate(row):
    climates = []

    for col, climate in climate_map.items():

        value = row[col]

        # Ignore empty cells / NaN / zeros
        if pd.notna(value) and value != 0:

            try:
                if float(value).is_integer():
                    value = int(value)
            except:
                pass

            climates.append(
                f"{climate} ({value})"
            )

    return " | ".join(climates)

# Create classification column
df["climatic moisture index"] = df.apply(
    classify_climate,
    axis=1
)

# Save output
df.to_csv(
    "CMI_tree_species.csv",
    index=False
)

print("Done. File saved as CMI_tree_species.csv")

Done. File saved as CMI_tree_species.csv


In [6]:
import pandas as pd

# Load datasets
flowering = pd.read_csv("/content/drive/MyDrive/flowering_trees.csv")
cmi = pd.read_csv("CMI_tree_species.csv")

# Clean species names
flowering["scientificname"] = (
    flowering["scientificname"]
    .astype(str)
    .str.strip()
    .str.lower()
)

cmi["species"] = (
    cmi["species"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Select columns to bring from CMI dataset
cols_to_merge = [
    "species",
    "n",
    "Q05",
    "Q95",
    "climatic moisture index"
]

cmi_subset = cmi[cols_to_merge]

# Merge while retaining all flowering tree species
merged = flowering.merge(
    cmi_subset,
    left_on="scientificname",
    right_on="species",
    how="left"
)

# Remove duplicate species column from CMI file
merged = merged.drop(columns=["species"])

# Save
merged.to_csv(
    "flowering_trees_CMI.csv",
    index=False
)

# Report match statistics
matched = merged["climatic moisture index"].notna().sum()
total = len(merged)

print(f"Matched: {matched}")
print(f"Unmatched: {total - matched}")
print(f"Total species: {total}")
print("Saved as flowering_trees_CMI.csv")

Matched: 42141
Unmatched: 11336
Total species: 53477
Saved as flowering_trees_CMI.csv


In [7]:
import pandas as pd
import re

# Load file
df = pd.read_csv("flowering_trees_CMI.csv")

def dominant_cmi(cmi_text):

    if pd.isna(cmi_text) or str(cmi_text).strip() == "":
        return cmi_text

    parts = str(cmi_text).split(" | ")

    cmi_scores = []

    for part in parts:
        match = re.search(r"\((\d+)\)\s*$", part)

        if match:
            cmi_scores.append((part, int(match.group(1))))

    if not cmi_scores:
        return cmi_text

    max_score = max(score for _, score in cmi_scores)

    # If at least one class has a score of 9,
    # keep only the classes with score 9
    if max_score == 9:
        dominant = [
            cmi_class
            for cmi_class, score in cmi_scores
            if score == 9
        ]
        return " | ".join(dominant)

    # Otherwise keep all classes
    return cmi_text

# Append new column
df["dominant climatic moisture index"] = (
    df["climatic moisture index"]
    .apply(dominant_cmi)
)

# Save
df.to_csv(
    "flowering_trees_dominant_CMI.csv",
    index=False
)

print("Done. Saved as flowering_trees_dominant_CMI.csv")

/tmp/ipykernel_2547/948097633.py:5: DtypeWarning: Columns (10,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("flowering_trees_CMI.csv")


Done. Saved as flowering_trees_dominant_CMI.csv


Biome/Tmo10

In [1]:
import pandas as pd

# Load file
df = pd.read_csv("/content/drive/MyDrive/TreeGOER_Tmo10_2024.txt", sep="|")

# Mapping of code columns to climate types
climate_map = {
    "A": "tropical (min temp coldest month 18°C or higher)",
    "C": "tropical (min temp coldest month less than 18°C)",
    "D": "subtropical",
    "E": "temperate",
    "F": "boreal",
    "A18": "polar"
}

def Biome(row):

    q05 = float(row["Q05"])
    q95 = float(row["Q95"])

    # Rule 1: both quantiles are zero
    if q05 == 0 and q95 == 0:
        return ""

    climate_list = []

    for col, climate in climate_map.items():

        value = row[col]

        # Rule 2: ignore polar if species is entirely in Tmo10 = 12 climates
        if col == "A18" and q05 == 12 and q95 == 12:
            continue

        # Ignore missing or zero values
        if pd.notna(value) and value != 0:

            try:
                if float(value).is_integer():
                    value = int(value)
            except:
                pass

            climate_list.append(
                f"{climate} ({value})"
            )

    return " | ".join(climate_list)

# Add new classification column
df["Biome"] = df.apply(
    Biome,
    axis=1
)

# Save output
df.to_csv(
    "Biome_Tree_Species.csv",
    index=False
)

print("Done.")
print("Saved as Biome_Tree_Species.csv")

Done.
Saved as Biome_Tree_Species.csv


In [2]:
import pandas as pd

# Load datasets
flowering = pd.read_csv("/content/drive/MyDrive/flowering_trees.csv")
tmo10 = pd.read_csv("Biome_Tree_Species.csv")

# Clean species names for better matching
flowering["scientificname"] = (
    flowering["scientificname"]
    .astype(str)
    .str.strip()
    .str.lower()
)

tmo10["species"] = (
    tmo10["species"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Columns to import from Tmo10 dataset
cols_to_merge = [
    "species",
    "n",
    "Q05",
    "Q95",
    "Biome"
]

tmo10_subset = tmo10[cols_to_merge]

# Merge while retaining all flowering species
merged = flowering.merge(
    tmo10_subset,
    left_on="scientificname",
    right_on="species",
    how="left"
)

# Remove duplicate species column
merged = merged.drop(columns=["species"])

# Optional: rename imported columns to avoid confusion
merged = merged.rename(columns={
    "n": "Tmo10_n",
    "Q05": "Tmo10_Q05",
    "Q95": "Tmo10_Q95"
})

# Save
merged.to_csv(
    "flowering_trees_biome.csv",
    index=False
)

# Match summary
matched = merged["Biome"].notna().sum()
total = len(merged)

print(f"Matched: {matched}")
print(f"Unmatched: {total - matched}")
print(f"Total species: {total}")
print("Saved as flowering_trees_Biome.csv")

Matched: 42111
Unmatched: 11366
Total species: 53477
Saved as flowering_trees_Biome.csv


In [3]:
import pandas as pd
import re

# Load file
df = pd.read_csv("flowering_trees_biome.csv")

def dominant_biome(biome_text):

    if pd.isna(biome_text) or str(biome_text).strip() == "":
        return biome_text

    parts = str(biome_text).split(" | ")

    biome_scores = []

    for part in parts:
        match = re.search(r"\((\d+)\)\s*$", part)

        if match:
            biome_scores.append((part, int(match.group(1))))

    if not biome_scores:
        return biome_text

    max_score = max(score for _, score in biome_scores)

    # If there is a 9, keep only the 9s
    if max_score == 9:
        dominant = [
            biome
            for biome, score in biome_scores
            if score == 9
        ]
        return " | ".join(dominant)

    # Otherwise keep all listed biomes
    return biome_text

# Create new column
df["dominant biome"] = df["Biome"].apply(dominant_biome)

# Save
df.to_csv(
    "flowering_trees_dominant_biome.csv",
    index=False
)

print("Done.")

/tmp/ipykernel_2547/4149977066.py:5: DtypeWarning: Columns (10,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("flowering_trees_biome.csv")


Done.
